# 08a -- FantasyPros Rankings

**Purpose**: Scrape FantasyPros dynasty rookie rankings and ingest into
`fact_rookie_rankings`. Self-contained â€” run independently of other source notebooks.

**Sources**:
| Source key | Format | URL |
|---|---|---|
| `FantasyPros_PPR` | Offense PPR | dynasty-rookies-overall.php |
| `FantasyPros_Superflex` | Offense Superflex | dynasty-rookies-superflex.php |

**IDP note**: FantasyPros IDP rookie rankings (`dynasty-rookies-idp.php`) render
client-side only — raw HTML `ecrData` returns a veteran draft board, not defensive
rookies. IDP is manually extracted and ingested via `08e_manual_rankings`.

**Workflow**:
1. Run cells top-to-bottom
2. If review file generated â†’ run `09_apply_fuzzy_review.ipynb` â†’ re-run ingestion cells

**Prerequisites**: `08_fact_rookie_rankings_seed.ipynb`, `01_dim_rookie_prospect.ipynb`

**Outputs**: `data/fact_rookie_rankings.parquet` (appended), `data/review_fuzzy_matches.csv` (if needed)

## 1 -- Setup & Shared Helpers

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import requests
import re
import json
import time
from pathlib import Path
from datetime import date, datetime
from dataclasses import dataclass, field
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from thefuzz import fuzz, process

# ---- Shared helpers + config from etl_helpers -------------------------------
import sys
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import (
    LeagueConfig, CFG, DATA, REVIEW, TODAY, ALIAS, DEFAULT_HEADERS,
    _make_session, _parse_rank_date, clean_player_name, generate_player_key,
    add_players_from_source, ingest_ranking_source, append_review,
)


In [14]:
# clean_player_name / generate_player_key / _parse_rank_date imported from etl_helpers

Helpers OK


## 2 -- add_players_from_source

In [ ]:
# add_players_from_source imported from etl_helpers (canonical, alias-aware)

## 3 -- ingest_ranking_source

In [ ]:
# ingest_ranking_source imported from etl_helpers

## 4 -- FantasyProsScraper

`ecrData` JSON is embedded directly in FantasyPros page HTML â€” no browser rendering
needed. The scraper extracts it with a regex + JSON parse.

**`grade`**: stored as `rank_ave` (simple average of expert ranks).

In [17]:
class FantasyProsScraper:
    # Scrapes ecrData JSON embedded in FantasyPros dynasty ranking pages.
    # ecrData key fields:
    #   rank_ecr  -> global_rank  (expert-weighted consensus rank)
    #   rank_ave  -> grade        (simple average across experts)
    #   pos_rank  -> positional_rank (parsed: "RB1" -> 1)

    _HEADERS = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
    }

    _POS_COMPAT = {
        "QB": {"QB"},
        "RB": {"RB", "HB", "FB"},
        "WR": {"WR"},
        "TE": {"TE"},
        "OL": {"OL", "OT", "OG", "C", "IOL", "G", "T"},
        "DL": {"DE", "DT", "EDGE", "NT", "DL", "DI", "ED"},
        "LB": {"LB", "ILB", "OLB", "MLB", "WILL", "SAM", "MIKE"},
        "DB": {"CB", "S", "FS", "SS", "DB", "SAF"},
        "ST": {"K", "P", "LS"},
    }

    SOURCES = {
        "FantasyPros_PPR": {
            "url": "https://www.fantasypros.com/nfl/rankings/dynasty-rookies-overall.php",
            "site_reference": "Offense - PPR",
            "source_site": "FantasyPros",
        },
        "FantasyPros_Superflex": {
            "url": "https://www.fantasypros.com/nfl/rankings/dynasty-rookies-superflex.php",
            "site_reference": "Offense - PPR Superflex",
            "source_site": "FantasyPros",
        },
    }

    def __init__(self, timeout: int = 30, delay: float = 2.0):
        self.timeout = timeout
        self.delay   = delay
        self.session = _make_session(timeout_sec=timeout)
        self.session.headers.update(self._HEADERS)

    def _fetch_ecr_data(self, url: str) -> dict:
        time.sleep(self.delay)
        resp = self.session.get(url, timeout=self.timeout)
        resp.raise_for_status()
        m = re.search(r"ecrData\s*=\s*(\{)", resp.text)
        if not m:
            raise ValueError(f"ecrData not found in page: {url}")
        raw = resp.text[m.start(1):]
        depth = end = 0
        for i, c in enumerate(raw):
            if c == "{":
                depth += 1
            elif c == "}":
                depth -= 1
                if depth == 0:
                    end = i + 1
                    break
        return json.loads(raw[:end])

    @staticmethod
    def _parse_pos_rank(pos_rank: str) -> int | None:
        m = re.search(r"\d+$", str(pos_rank or ""))
        return int(m.group()) if m else None


    def scrape(self, source_key: str) -> pd.DataFrame:
        cfg  = self.SOURCES[source_key]
        data = self._fetch_ecr_data(cfg["url"])
        players = data.get("players", [])
        if not players:
            raise ValueError(f"No players in ecrData for {cfg['url']}")

        rows = []
        for p in players:
            rows.append({
                "player_name":     p["player_name"],
                "position_raw":    p.get("player_position_id", ""),
                "school_raw":      "",
                "nfl_team":        p.get("player_team_id", ""),
                "fp_player_id":    p.get("player_id"),
                "global_rank":     p.get("rank_ecr"),
                "positional_rank": self._parse_pos_rank(p.get("pos_rank")),
                "grade":           float(p["rank_ave"]) if p.get("rank_ave") else None,
                "rank_std":        float(p["rank_std"]) if p.get("rank_std") else None,
                "tier":            p.get("tier"),
            })

        df = pd.DataFrame(rows)


        meta = data.get
        rank_date = _parse_rank_date(meta('last_updated'))
        print(f"Scraped {source_key}: {len(df)} players | type={meta('type')} | "
              f"experts={meta('total_experts')} | updated={meta('last_updated')} -> rank_date={rank_date}")
        return df, rank_date

## 5 -- Run FantasyPros Ingestion

In [18]:
SOURCES = FantasyProsScraper.SOURCES  # alias for validation cell

SCRAPER = FantasyProsScraper(timeout=30, delay=2.0)
PHASE        = "post_draft"
_failed      = []
_all_reviews = []  # accumulate review rows across all sources; written once at end

for source_key, cfg in FantasyProsScraper.SOURCES.items():
    print("\n" + "=" * 60)
    print(f"Source : {source_key}  ({cfg['site_reference']})")
    try:
        rankings_df, rank_date = SCRAPER.scrape(source_key)
    except Exception as e:
        print(f"  ERROR scraping {source_key}: {e}")
        _failed.append(source_key)
        continue

    try:
        _dim_rp, review = add_players_from_source(
            rankings_df, source_name=source_key, draft_year=CFG.draft_year
        )
    except Exception as e:
        print(f"  ERROR in add_players_from_source: {e}")
        _failed.append(source_key)
        continue

    if not review.empty:
        # Accumulate -- do NOT skip ingestion; review players are logged as unmatched below
        print(f"  -> {len(review)} players flagged for review (accumulated across sources)")
        _all_reviews.append(review)

    try:
        ingest_ranking_source(
            rankings_df,
            source_name=source_key,
            source_site=cfg["source_site"],
            phase=PHASE,
            draft_year=CFG.draft_year,
            rank_date=rank_date,
        )
    except Exception as e:
        print(f"  ERROR ingesting {source_key}: {e}")
        _failed.append(source_key)

# -- Write combined review file if any players need manual decisions ----------
if _all_reviews:
    new_review = pd.concat(_all_reviews, ignore_index=True)
    rp = DATA / "review" / "review_fuzzy_matches.csv"
    if rp.exists():
        existing = pd.read_csv(rp)
        combined = pd.concat([existing, new_review], ignore_index=True)
        # keep="first" preserves already-filled action values from prior review sessions
        combined.drop_duplicates(subset=["new_name", "source"], keep="first", inplace=True)
        n_added = len(combined) - len(existing)
        combined.to_csv(rp, index=False)
        print("\n" + "=" * 60)
        print(f"Review file updated: +{max(n_added, 0)} new rows ({len(combined)} total) -> {rp}")
    else:
        new_review.to_csv(rp, index=False)
        print("\n" + "=" * 60)
        print(f"Review file: {rp}  ({len(new_review)} rows)")
    print("\nFill 'action' column ('match' or 'new'), run 09_apply_fuzzy_review, then re-run.")
if _failed:
    print("\nFailed sources (re-run individually):", _failed)
else:
    print("\nAll FantasyPros sources completed.")


Source : FantasyPros_PPR  (Offense - PPR)
Scraped FantasyPros_PPR: 145 players | type=Rookies | experts=44 | updated=5/16 -> rank_date=5/16
Source: FantasyPros_PPR
  Already in dim_rookie_prospect : 127
  Auto-matched (>=90)       : 3
  New prospects added             : 0
  Needs manual review             : 15
  -> 15 players flagged for review (accumulated across sources)
WARN: 18 players pending review -- will be picked up after apply_review_decisions():
  Omar Cooper Jr.
  Nick Singleton
  De'Zhaun Stribling
  Eli Raridon
  Kevin Coleman Jr.
  Cyrus Allen
  Matt Hibner
  Dean Connors
  Harrison Wallace III
  Emmanuel Henderson Jr.
  ... and 8 more
Ingested: 127 rows | source=FantasyPros_PPR | site=FantasyPros | phase=post_draft
fact_rookie_rankings total: 127

Source : FantasyPros_Superflex  (Offense - PPR Superflex)
Scraped FantasyPros_Superflex: 157 players | type=Rookies | experts=44 | updated=5/16 -> rank_date=5/16
Source: FantasyPros_Superflex
  Already in dim_rookie_prospect 

## 6 -- Validation

In [19]:
# Mini-validation: rows written by this source
import pandas as pd
from pathlib import Path
fr = pd.read_parquet(Path(CFG.data_dir) / "fact_rookie_rankings.parquet")
src_keys = list(SOURCES.keys())
sub = fr[fr["source_name"].isin(src_keys)]
print(f"Rows for this source: {len(sub)}")
print(sub.groupby(["source_name", "phase"]).size().to_string())
print()
print(f"fact_rookie_rankings total: {len(fr)} rows")
dupes = fr.duplicated(subset=["player_key","source_name","phase","draft_year"]).sum()
print(f"Composite key duplicates: {dupes} (expected 0)")

Rows for this source: 309
source_name            phase     
FantasyPros_IDP        post_draft     45
FantasyPros_PPR        post_draft    127
FantasyPros_Superflex  post_draft    137

fact_rookie_rankings total: 309 rows
Composite key duplicates: 0 (expected 0)
